In [ ]:
# Pipeline de Codificación
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from pathlib import Path

def encode_feature(df) -> tuple:
    print('🔡 Codificando el dataset')
    meal_info_encoder = OneHotEncoder()
    city_name_encoder = OrderCategoryEncoder()
    meal_name_encoder = OrderCategoryEncoder()
    df = apply_ohe_encoding(df, meal_info_encoder)
    df = apply_ordinal_encoding(df, city_name_encoder, meal_name_encoder)
    print('✅ Codificación del dataset completada')
    print('-'*40)
    
    artifacts = {}
    artifacts['meal_info'] = meal_info_encoder
    artifacts['city_name'] = city_name_encoder
    artifacts['meal_name'] = meal_name_encoder
    
    return df, artifacts
    
class OrderCategoryEncoder():
    
    def __init__(self):
        self.categories_: list[str] = None
        self.category_map: dict[str: int] = None
        self.inverse_category_map: dict[int: str] = None
        
    def fit(self, ordered_categories: list[str]):
        self.categories_ = ordered_categories
        self.category_map = {category: i+1 for i, category in enumerate(ordered_categories)}
        self.inverse_category_map = {i+1: category for i, category in enumerate(ordered_categories)}
    
    def transform(self, values: list[str]) -> np.array:
        if set(values) - set(self.category_map):
            for value in values:
                if value not in self.category_map:
                    print(f"⚠️ '{value}' no se encontró en los datos de entrenamiento")
            raise ValueError('Se encontró una nueva categoría, no se puede transformar')
        return np.array([self.category_map[value] for value in values])
        
    def inverse_transform(self, values: list[str]) -> np.array:
        if set(values) - set(self.inverse_category_map):
            for value in values:
                if value not in self.inverse_category_map:
                    print(f"⚠️ '{value}' no se encontró en los datos de entrenamiento")
            raise ValueError('Se encontró una nueva categoría, no se puede transformar')
        return np.array([self.inverse_category_map[value] for value in values])

def get_ohe_column(ohe_encoder):
    columns = []
    for a in ohe_encoder.categories_:
        columns.extend(list(a))
    return columns

def apply_ohe_encoding(df, meal_info_encoder):
    print('🔢 Aplicando one-hot-encoding a meal_category y meal_type')
    ohe_vals = meal_info_encoder.fit_transform(df[['meal_category', 'meal_type']].values).toarray()
    df_ohe = pd.DataFrame(ohe_vals, columns=get_ohe_column(meal_info_encoder)).astype('int')
    df = pd.concat((df.reset_index(drop=True), df_ohe.reset_index(drop=True)), axis=1)
    df = df.drop(columns=['meal_category', 'meal_type'])
    return df

def apply_ordinal_encoding(df, city_name_encoder, meal_name_encoder):
    print('🏙️ Aplicando codificación categórica ordenada a city_name')
    ordered_city_names = (
        df.groupby('city_name')['num_orders'].sum().sort_values()
        .reset_index('city_name').drop(columns='num_orders')
        .reset_index().set_index('city_name').to_dict()['index']
    )
    city_name_encoder.fit(ordered_city_names)
    df['city_id'] = city_name_encoder.transform(df['city_name'].values)
    
    print('🍽️ Aplicando codificación categórica ordenada a meal_name')
    ordered_meal_name = (
        df.groupby('meal_name')['num_orders'].sum().sort_values()
        .reset_index('meal_name').drop(columns='num_orders')
        .reset_index().set_index('meal_name').to_dict()['index']
    )
    meal_name_encoder.fit(ordered_meal_name)
    df['meal_id'] = meal_name_encoder.transform(df['meal_name'].values)
    
    df = df.drop(columns=['city_name', 'meal_name', 'Unnamed: 0'])
    
    return df

def load_meal_demand_dataset(file_name, data_path='../data/'):
    print('Loading dataset')
    file_path = Path(data_path)/file_name
    df = pd.read_csv(file_path)
    print(f'Loaded {len(df)} rows from {file_path}')
    print(df.dtypes)
    print(df.describe())
    print(df)
    print('Done loading dataset')
    print('-'*20)
    return df

df = load_meal_demand_dataset('new_meal_historical.csv')
df, encoders = encode_feature(df)
df

Loading dataset
Loaded 446732 rows from ../data/new_meal_historical.csv
Unnamed: 0                 int64
week_number                int64
checkout_price           float64
base_price               float64
emailer_for_promotion      int64
homepage_featured          int64
num_orders                 int64
op_area                  float64
city_name                 object
meal_name                 object
meal_category             object
meal_type                 object
dtype: object
          Unnamed: 0    week_number  checkout_price     base_price  \
count  446732.000000  446732.000000   446732.000000  446732.000000   
mean   227887.226550      74.640200      332.163814     354.064719   
std    131899.583459      41.558576      152.900341     160.675874   
min         0.000000       1.000000        2.970000      55.350000   
25%    113421.750000      39.000000      228.980000     243.500000   
50%    227716.500000      75.000000      296.820000     310.460000   
75%    342173.250000     111

,week_number,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,op_area,Meat,Other,Seafood,Vegetarian,beverage,dessert,main,side,starter,city_id,meal_id
0,1,136.83,152.29,0,0,177,2.0,0,0,0,1,0,0,1,0,0,68,2
1,1,136.83,135.83,0,0,270,2.0,1,0,0,0,0,0,1,0,0,68,4
2,1,134.86,135.86,0,0,189,2.0,1,0,0,0,0,0,1,0,0,68,15
3,1,339.50,437.53,0,0,54,2.0,0,0,0,1,0,0,0,0,1,68,44
4,1,243.50,242.50,0,0,40,2.0,0,0,1,0,0,0,1,0,0,68,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446727,145,484.09,484.09,0,0,68,4.5,1,0,0,0,0,0,1,0,0,44,35
446728,145,482.09,482.09,0,0,42,4.5,0,1,0,0,0,1,0,0,0,44,37
446729,145,237.68,321.07,0,0,501,4.5,1,0,0,0,0,0,1,0,0,44,11
446730,145,243.50,313.34,0,0,729,4.5,0,1,0,0,0,1,0,0,0,44,13
